In [1]:
# 4.8 Part 1 – Customer Data Wrangling

# 1. Import libraries and customer data
import pandas as pd
import numpy as np
import os

# Set base path to main project folder
path = r"/Users/a/Documents/Instacart Basket Analysis"

# Path to customers data in Original Data folder
cust_path = os.path.join(path, "02 Data", "Original Data", "customers.csv")

# Import customers dataframe
df_cust = pd.read_csv(cust_path)

# Quick preview and shape check
df_cust.head(), df_cust.shape

(   user_id First Name    Surnam  Gender       STATE  Age date_joined  \
 0    26711    Deborah  Esquivel  Female    Missouri   48    1/1/2017   
 1    33890   Patricia      Hart  Female  New Mexico   36    1/1/2017   
 2    65803    Kenneth    Farley    Male       Idaho   35    1/1/2017   
 3   125935   Michelle     Hicks  Female        Iowa   40    1/1/2017   
 4   130797        Ann   Gilmore  Female    Maryland   26    1/1/2017   
 
    n_dependants fam_status  income  
 0             3    married  165665  
 1             0     single   59285  
 2             2    married   99568  
 3             0     single   42049  
 4             1    married   40374  ,
 (206209, 10))

In [4]:
# 2. Basic wrangling: rename columns for consistency

df_cust.rename(columns={
    'First Name': 'first_name',
    'Surname': 'last_name',
    'Gender': 'gender',
    'STATE': 'state',
    'Age': 'age',
    'n_dependants': 'num_dependents',
    'fam_status': 'marital_status',
    'income': 'income',
    'date_joined': 'date_joined'
}, inplace=True)

# Quick check
df_cust.head()

,user_id,first_name,Surnam,gender,state,age,date_joined,num_dependents,marital_status,income
0,26711,Deborah,Esquivel,Female,Missouri,48,1/1/2017,3,married,165665
1,33890,Patricia,Hart,Female,New Mexico,36,1/1/2017,0,single,59285
2,65803,Kenneth,Farley,Male,Idaho,35,1/1/2017,2,married,99568
3,125935,Michelle,Hicks,Female,Iowa,40,1/1/2017,0,single,42049
4,130797,Ann,Gilmore,Female,Maryland,26,1/1/2017,1,married,40374


In [6]:
# 3. Check for missing values
df_cust.isnull().sum()

user_id               0
first_name        11259
Surnam                0
gender                0
state                 0
age                   0
date_joined           0
num_dependents        0
marital_status        0
income                0
dtype: int64

In [13]:
# 4. Check for duplicates
df_cust.duplicated().sum()

0

In [15]:
# 5. Drop first_name column (too many missing values, not useful for analysis)
df_cust = df_cust.drop(columns=['first_name'], errors='ignore')

# Convert data types
df_cust['date_joined'] = pd.to_datetime(df_cust['date_joined'])

# Ensure user_id is the same type as in Instacart data (usually int)
df_cust['user_id'] = df_cust['user_id'].astype('int64')

# Quick check
df_cust.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206209 entries, 0 to 206208
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   user_id         206209 non-null  int64         
 1   Surnam          206209 non-null  object        
 2   gender          206209 non-null  object        
 3   state           206209 non-null  object        
 4   age             206209 non-null  int64         
 5   date_joined     206209 non-null  datetime64[ns]
 6   num_dependents  206209 non-null  int64         
 7   marital_status  206209 non-null  object        
 8   income          206209 non-null  int64         
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 14.2+ MB


In [2]:

# Path to prepared Instacart data (latest version)
insta_path = os.path.join(path, "02 Data", "Prepared Data", "ords_prods_grouped.pkl")

# Import Instacart dataframe
df_ords = pd.read_pickle(insta_path)

# Quick preview and shape check
df_ords.head(), df_ords.shape

(   order_id  user_id  order_number  order_dow  order_hour_of_day  \
 0   2539329        1             1          2                  8   
 1   2539329        1             1          2                  8   
 2   2539329        1             1          2                  8   
 3   2539329        1             1          2                  8   
 4   2539329        1             1          2                  8   
 
    days_since_prior_order  product_id  add_to_cart_order  reordered  \
 0                     0.0         196                  1          0   
 1                     0.0       14084                  2          0   
 2                     0.0       12427                  3          0   
 3                     0.0       26088                  4          0   
 4                     0.0       26405                  5          0   
 
                               product_name  ...  price_label  busiest_day  \
 0                                     Soda  ...          Mid         Sl

In [19]:
# 7. Merge customer data with Instacart data on user_id

df_merged = df_ords.merge(df_cust, on='user_id', how='left')

# Quick check
df_merged.head(), df_merged.shape

(   order_id  user_id  order_number  order_dow  order_hour_of_day  \
 0   2539329        1             1          2                  8   
 1   2539329        1             1          2                  8   
 2   2539329        1             1          2                  8   
 3   2539329        1             1          2                  8   
 4   2539329        1             1          2                  8   
 
    days_since_prior_order  product_id  add_to_cart_order  reordered  \
 0                     0.0         196                  1          0   
 1                     0.0       14084                  2          0   
 2                     0.0       12427                  3          0   
 3                     0.0       26088                  4          0   
 4                     0.0       26405                  5          0   
 
                               product_name  ...  median_days_between_orders  \
 0                                     Soda  ...                      

In [21]:
# 8. Post-merge quality checks

# Check for missing customer data after merge
df_merged[['gender', 'state', 'age', 'income']].isnull().sum()

# Check duplicates (unlikely but required)
df_merged.duplicated().sum()

# Quick info summary
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32436241 entries, 0 to 32436240
Data columns (total 31 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   order_id                    int64         
 1   user_id                     int64         
 2   order_number                int64         
 3   order_dow                   int64         
 4   order_hour_of_day           int64         
 5   days_since_prior_order      float64       
 6   product_id                  int64         
 7   add_to_cart_order           int64         
 8   reordered                   int64         
 9   product_name                object        
 10  aisle_id                    float64       
 11  department_id               float64       
 12  prices                      float64       
 13  price_label                 object        
 14  busiest_day                 object        
 15  busiest_days                object        
 16  busiest_period_o

In [23]:
# 9. Export merged dataframe for Part 2
export_path = os.path.join(path, "02 Data", "Prepared Data", "ords_prods_customers.pkl")
df_merged.to_pickle(export_path)

export_path

'/Users/a/Documents/Instacart Basket Analysis/02 Data/Prepared Data/ords_prods_customers.pkl'